In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# **LOAD DATA FRAME**

---



In [ ]:
import pandas as pd
# Load the .dat file into a DataFrame
file_path = '/content/drive/MyDrive/Gaussian_Integral/ATLAS_GI_IOresult.dat'
data = pd.read_csv(file_path, delim_whitespace=True, header=None, engine='python')
print(data.head())
print(data.iloc[:, -3:].head())
import pandas as pd

# Set display options to show all columns in a single line
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Adjust width as needed

# Display the DataFrame
print(data.head())  # This will show the first few rows with all columns in a single line
# Sobrescribe el DataFrame original eliminando duplicados en columna 1
data = data.drop_duplicates(subset=1, keep='first').reset_index(drop=True)

# Verificamos la nueva forma
print("Filas luego de eliminar duplicados:", len(data))
print("IDs únicos en col 1:", data[1].nunique())



/tmp/ipython-input-4156321417.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(file_path, delim_whitespace=True, header=None, engine='python')


   0     1  2       3       4      5        6       7       8       9   ...  \
0   1  1a62  A  1.2970  0.4236  3.645  0.02173 -0.3509  1.2360  1.6110  ...   
1   2  1ab1  A  0.4890  0.9617  2.546  0.51950  0.5938  0.6029  0.6019  ...   
2   3  1ah7  A  2.6040  2.3810  5.383  2.49100  3.4620  3.0130  3.5740  ...   
3   4  1ail  A  0.7441  2.9330  3.353  3.17300  2.1860  2.3270  1.4560  ...   
4   5  1aol  A  2.4240  0.5442  4.530  0.16640  0.7012  0.4889  2.0390  ...   

        25       26      27      28      29       30       31      32    33  \
0  0.12740  0.12250 -0.4103 -0.1015  0.1582  0.06054  0.02185 -0.1065  1.37   
1 -0.69990  0.54800 -0.5913 -1.4960  5.8380 -2.31500 -3.79100 -2.0550  0.66   
2  0.05350  0.01457  0.6531 -0.2520 -0.1576  1.03500  0.46810  0.2457  0.68   
3  0.86000  1.64100  3.3210  0.9377  4.6480  5.41800  6.74300  5.0480  1.42   
4  0.04921 -0.09331  0.1015 -0.1209 -0.2470 -0.26600 -0.29280  0.3713  1.05   

   34  
0   1  
1   0  
2   0  
3   1  
4   0  

[

# **GENERATE TENSORS/DATALOADERS**

---




In [ ]:
from torch.utils.data import Dataset
import torch

class ProteinDataset(Dataset):
    def __init__(self, data, id_list, feature_cols=range(3, 33), label_col=34):
        self.data = data[data[1].isin(id_list)].copy()
        self.X = self.data.iloc[:, feature_cols].values.astype('float32')  # (N, 30)
        self.y = self.data.iloc[:, label_col].values.astype('float32')     # (N,)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].reshape(1, -1)  # (1, 30)
        y = self.y[idx]
        return torch.tensor(x), torch.tensor([y])


In [ ]:
from torch.utils.data import DataLoader

def get_dataloaders_from_json_fold(fold_id, json_path, data, hyperparams):
    with open(json_path, "r") as f:
        folds = json.load(f)

    fold = folds['folds'][fold_id]
    train_ids = fold['train_ids']
    val_ids   = fold['val_ids']

    train_dataset = ProteinDataset(data, train_ids)
    val_dataset   = ProteinDataset(data, val_ids)

    train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

    return train_loader, val_loader


In [ ]:
HYPERPARAMS = {
    'input_dim': 1,             # Number of input channels, adjusted for reshaped data
    'hidden_dim': 128,          # Number of filters in convolutional layers
    'output_dim': 1,            # Binary classification (1 output unit)
    'kernel_size': 3,           # Smaller kernel size to match no dilation and reduce overfitting
    'batch_size': 32,           # Same as before
    'learning_rate': 0.001,     # Standard learning rate, tune if necessary
    'num_epochs': 50,           # Number of training epochs, set for early stopping
    'k_folds': 5,               # Number of cross-validation folds
    'patience': 7,              # Early stopping patience
    'weight_decay': 1e-4        # L2 regularization to prevent overfitting
}

In [ ]:
import json
NUM_FOLDS = 5  # Ajustar si usaste otro número

all_dataloaders = []

for fold_id in range(NUM_FOLDS):
    train_loader, val_loader = get_dataloaders_from_json_fold(
        fold_id=fold_id,
        json_path="/content/drive/MyDrive/Gaussian_Integral/folds_gaussian_integral_ids_clean.json",
        data=data,
        hyperparams=HYPERPARAMS
    )

    print(f"Fold {fold_id} - Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    all_dataloaders.append({
        "fold": fold_id,
        "train": train_loader,
        "val": val_loader
    })


Fold 0 - Train batches: 28, Val batches: 7
Fold 1 - Train batches: 28, Val batches: 7
Fold 2 - Train batches: 28, Val batches: 7
Fold 3 - Train batches: 28, Val batches: 7
Fold 4 - Train batches: 28, Val batches: 7


In [ ]:
for fold_dict in all_dataloaders:
    fold_id = fold_dict["fold"]
    train_loader = fold_dict["train"]

    x_batch, y_batch = next(iter(train_loader))
    print(f"Fold {fold_id} - x_batch shape: {x_batch.shape}, y_batch shape: {y_batch.shape}")


Fold 0 - x_batch shape: torch.Size([32, 1, 30]), y_batch shape: torch.Size([32, 1])
Fold 1 - x_batch shape: torch.Size([32, 1, 30]), y_batch shape: torch.Size([32, 1])
Fold 2 - x_batch shape: torch.Size([32, 1, 30]), y_batch shape: torch.Size([32, 1])
Fold 3 - x_batch shape: torch.Size([32, 1, 30]), y_batch shape: torch.Size([32, 1])
Fold 4 - x_batch shape: torch.Size([32, 1, 30]), y_batch shape: torch.Size([32, 1])


# **DEFINE NET**

---



In [ ]:
import torch
import torch.nn as nn

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super(AttentionLayer, self).__init__()
        self.attention_weights = nn.Linear(hidden_dim, 1)  # Learnable weights
        self.softmax = nn.Softmax(dim=1)  # Softmax over the time dimension

    def forward(self, x):
        # x: (batch_size, hidden_dim, seq_length)
        attn_scores = self.attention_weights(x.permute(0, 2, 1))  # (batch_size, seq_length, 1)
        attn_scores = self.softmax(attn_scores.squeeze(-1))  # (batch_size, seq_length)
        attn_applied = torch.bmm(x, attn_scores.unsqueeze(-1)).squeeze(-1)  # Weighted sum: (batch_size, hidden_dim)
        return attn_applied


class ProteinClassifierTCNWithAttentionNoDilation(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, kernel_size=3):
        super(ProteinClassifierTCNWithAttentionNoDilation, self).__init__()

        # First convolutional layer
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu1 = nn.ReLU()

        # Second convolutional layer
        self.conv2 = nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu2 = nn.ReLU()

        # Third convolutional layer
        self.conv3 = nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu3 = nn.ReLU()

        # Add an extra convolutional layer
        self.conv4 = nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu4 = nn.ReLU()

        # Attention layer
        self.attention = AttentionLayer(hidden_dim)

        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Temporal convolution layers
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.relu3(self.conv3(x))
        x = self.relu4(self.conv4(x))

        # Attention mechanism
        x = self.attention(x)  # Weighted feature aggregation

        # Fully connected layer for binary classification
        x = self.sigmoid(self.fc(x))
        return x

# **TRAIN AND VALIDATE NET**

---


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import numpy as np

def train_model(model, train_loader, val_loader, hyperparams, device='cuda'):
    criterion = nn.BCELoss() #Define a loss function of Binary Cross Entropy for the classification task.
    optimizer = optim.Adam(model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay']) #Define the optimizer which will update the weights of the model using gradients
    model.to(device)

    best_val_loss = float('inf') #Start saving validation loss metrics to find the model with la menor perdida alcanzada
    patience_counter = 0
    best_model_wts = model.state_dict()

    for epoch in range(hyperparams['num_epochs']):
        model.train()
        train_losses, y_true_train, y_pred_train = [], [], []

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            y_true_train += y_batch.cpu().numpy().flatten().tolist()
            y_pred_train += (outputs.detach().cpu().numpy().flatten() > 0.5).tolist()

        train_acc = accuracy_score(y_true_train, y_pred_train)

        # Validation
        model.eval()
        val_losses, y_true_val, y_pred_val = [], [], []
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(device), y_val.to(device)
                val_out = model(x_val)
                loss = criterion(val_out, y_val)
                val_losses.append(loss.item())
                y_true_val += y_val.cpu().numpy().flatten().tolist()
                y_pred_val += (val_out.cpu().numpy().flatten() > 0.5).tolist()

        val_loss = np.mean(val_losses)
        val_acc = accuracy_score(y_true_val, y_pred_val)

        print(f"Epoch {epoch+1} — Train Acc: {train_acc:.3f}, Val Acc: {val_acc:.3f}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= hyperparams['patience']:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    model.load_state_dict(best_model_wts)
    return model


In [ ]:
def train_and_validate_kfold(all_dataloaders, hyperparams):
    trained_models = []
    val_accuracies = []

    for fold in range(len(all_dataloaders)):
        print(f"\n🟦 Fold {fold}")
        train_loader = all_dataloaders[fold]["train"]
        val_loader = all_dataloaders[fold]["val"]

        model = ProteinClassifierTCNWithAttentionNoDilation(
            input_dim=hyperparams['input_dim'],
            hidden_dim=hyperparams['hidden_dim'],
            output_dim=hyperparams['output_dim'],
            kernel_size=hyperparams['kernel_size']
        )

        trained_model = train_model(model, train_loader, val_loader, hyperparams)
        trained_models.append(trained_model)

        # Eval final
        y_true, y_pred = [], []
        trained_model.eval()
        with torch.no_grad():
            for x_val, y_val in val_loader:
                out = trained_model(x_val.cuda())
                y_true += y_val.cpu().numpy().flatten().tolist()
                y_pred += (out.cpu().numpy().flatten() > 0.5).tolist()

        val_acc = accuracy_score(y_true, y_pred)
        val_accuracies.append(val_acc)
        print(f"✅ Fold {fold} finished — Final Val Acc: {val_acc:.3f}")

    # Summary
    for fold, acc in enumerate(val_accuracies):
        print(f"Fold {fold} — Val Acc: {acc:.3f}")
    print(f"\n📈 Average Val Acc: {np.mean(val_accuracies):.3f}")

    return trained_models, val_accuracies


In [ ]:
train_and_validate_kfold(all_dataloaders, HYPERPARAMS)



🟦 Fold 0
Epoch 001 | train_loss=0.6919 | val_loss=0.6920 | train_acc=0.502 | val_acc=0.550
Epoch 002 | train_loss=0.6855 | val_loss=0.6901 | train_acc=0.552 | val_acc=0.500
Epoch 003 | train_loss=0.6724 | val_loss=0.6925 | train_acc=0.565 | val_acc=0.564
Epoch 004 | train_loss=0.6525 | val_loss=0.6819 | train_acc=0.601 | val_acc=0.545
Epoch 005 | train_loss=0.6478 | val_loss=0.6829 | train_acc=0.602 | val_acc=0.595
Epoch 006 | train_loss=0.6373 | val_loss=0.6877 | train_acc=0.626 | val_acc=0.582
Epoch 007 | train_loss=0.6324 | val_loss=0.7250 | train_acc=0.630 | val_acc=0.527
Epoch 008 | train_loss=0.6273 | val_loss=0.6741 | train_acc=0.645 | val_acc=0.627
Epoch 009 | train_loss=0.6195 | val_loss=0.6525 | train_acc=0.660 | val_acc=0.627
Epoch 010 | train_loss=0.6043 | val_loss=0.7118 | train_acc=0.661 | val_acc=0.500
Epoch 011 | train_loss=0.6179 | val_loss=0.6450 | train_acc=0.645 | val_acc=0.636
Epoch 012 | train_loss=0.5871 | val_loss=0.6365 | train_acc=0.685 | val_acc=0.645
Epoch 

([ProteinClassifierTCNWithAttentionNoDilation(
    (conv1): Conv1d(1, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (relu1): ReLU()
    (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (relu2): ReLU()
    (conv3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (relu3): ReLU()
    (conv4): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (relu4): ReLU()
    (attention): AttentionLayer(
      (attention_weights): Linear(in_features=128, out_features=1, bias=True)
      (softmax): Softmax(dim=1)
    )
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (sigmoid): Sigmoid()
  ),
  ProteinClassifierTCNWithAttentionNoDilation(
    (conv1): Conv1d(1, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (relu1): ReLU()
    (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (relu2): ReLU()
    (conv3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (relu3): ReLU()
    (

**RE-TRAIN BEST PERFORMING MODEL**
---



In [ ]:
fold3_train_ids = folds_dict['folds'][3]['train_ids']
fold3_val_ids   = folds_dict['folds'][3]['val_ids']

combined_ids = fold3_train_ids + fold3_val_ids
print(f"Total train+val IDs (fold 3): {len(combined_ids)}")
# Paso 2: Subsetear el DataFrame
subset_df = data.loc[data[1].isin(combined_ids)].reset_index(drop=True)

Total train+val IDs (fold 3): 1099


In [ ]:
subset_df.shape

(1099, 35)

In [ ]:
features = subset_df.iloc[:, 3:33].values  # Columnas 3 a 32 = 30 features
labels   = subset_df.iloc[:, 34].values    # Columna 34 = etiquetas binarias

import torch
X_tensor = torch.tensor(features, dtype=torch.float32).unsqueeze(1)  # [N, 1, 30]
y_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)    # [N, 1]

from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = HYPERPARAMS['batch_size']
dataset = TensorDataset(X_tensor, y_tensor)
final_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)



In [ ]:
model_final = ProteinClassifierTCNWithAttentionNoDilation(
    input_dim=HYPERPARAMS['input_dim'],
    hidden_dim=HYPERPARAMS['hidden_dim'],
    output_dim=HYPERPARAMS['output_dim'],
    kernel_size=HYPERPARAMS['kernel_size']
)

trained_final_model = train_model(model_final, final_loader, final_loader, HYPERPARAMS)


Epoch 1 — Train Acc: 0.524, Val Acc: 0.514, Val Loss: 0.6850
Epoch 2 — Train Acc: 0.587, Val Acc: 0.590, Val Loss: 0.6838
Epoch 3 — Train Acc: 0.574, Val Acc: 0.604, Val Loss: 0.6651
Epoch 4 — Train Acc: 0.607, Val Acc: 0.585, Val Loss: 0.6552
Epoch 5 — Train Acc: 0.591, Val Acc: 0.585, Val Loss: 0.6540
Epoch 6 — Train Acc: 0.615, Val Acc: 0.622, Val Loss: 0.6333
Epoch 7 — Train Acc: 0.642, Val Acc: 0.700, Val Loss: 0.6086
Epoch 8 — Train Acc: 0.631, Val Acc: 0.651, Val Loss: 0.6209
Epoch 9 — Train Acc: 0.669, Val Acc: 0.701, Val Loss: 0.5897
Epoch 10 — Train Acc: 0.680, Val Acc: 0.699, Val Loss: 0.5733
Epoch 11 — Train Acc: 0.655, Val Acc: 0.683, Val Loss: 0.6118
Epoch 12 — Train Acc: 0.661, Val Acc: 0.634, Val Loss: 0.6050
Epoch 13 — Train Acc: 0.691, Val Acc: 0.725, Val Loss: 0.5520
Epoch 14 — Train Acc: 0.688, Val Acc: 0.590, Val Loss: 0.6798
Epoch 15 — Train Acc: 0.704, Val Acc: 0.643, Val Loss: 0.6143
Epoch 16 — Train Acc: 0.705, Val Acc: 0.730, Val Loss: 0.5203
Epoch 17 — Train 

In [ ]:
# Suponiendo que 'final_model' es tu modelo entrenado desde cero
#model_path = "/content/drive/MyDrive/Gaussian_Integral/trained_final_model.pt"
torch.save(trained_final_model.state_dict(), model_path)
print(f"Modelo guardado en: {model_path}")


Modelo guardado en: /content/drive/MyDrive/Gaussian_Integral/trained_final_model.pt


# **TEST MODEL!**

---



In [ ]:
HYPERPARAMS = {
    'input_dim': 1,             # Number of input channels, adjusted for reshaped data
    'hidden_dim': 128,          # Number of filters in convolutional layers
    'output_dim': 1,            # Binary classification (1 output unit)
    'kernel_size': 3,           # Smaller kernel size to match no dilation and reduce overfitting
    'batch_size': 32,           # Same as before
    'learning_rate': 0.001,     # Standard learning rate, tune if necessary
    'num_epochs': 50,           # Number of training epochs, set for early stopping
    'k_folds': 5,               # Number of cross-validation folds
    'patience': 7,              # Early stopping patience
    'weight_decay': 1e-4        # L2 regularization to prevent overfitting
}

In [ ]:
import torch
import torch.nn as nn

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super(AttentionLayer, self).__init__()
        self.attention_weights = nn.Linear(hidden_dim, 1)  # Learnable weights
        self.softmax = nn.Softmax(dim=1)  # Softmax over the time dimension

    def forward(self, x):
        # x: (batch_size, hidden_dim, seq_length)
        attn_scores = self.attention_weights(x.permute(0, 2, 1))  # (batch_size, seq_length, 1)
        attn_scores = self.softmax(attn_scores.squeeze(-1))  # (batch_size, seq_length)
        attn_applied = torch.bmm(x, attn_scores.unsqueeze(-1)).squeeze(-1)  # Weighted sum: (batch_size, hidden_dim)
        return attn_applied


class ProteinClassifierTCNWithAttentionNoDilation(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, kernel_size=3):
        super(ProteinClassifierTCNWithAttentionNoDilation, self).__init__()

        # First convolutional layer
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu1 = nn.ReLU()

        # Second convolutional layer
        self.conv2 = nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu2 = nn.ReLU()

        # Third convolutional layer
        self.conv3 = nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu3 = nn.ReLU()

        # Add an extra convolutional layer
        self.conv4 = nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim,
                               kernel_size=kernel_size, dilation=1, padding=kernel_size // 2)
        self.relu4 = nn.ReLU()

        # Attention layer
        self.attention = AttentionLayer(hidden_dim)

        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Temporal convolution layers
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.relu3(self.conv3(x))
        x = self.relu4(self.conv4(x))

        # Attention mechanism
        x = self.attention(x)  # Weighted feature aggregation

        # Fully connected layer for binary classification
        x = self.sigmoid(self.fc(x))
        return x

In [ ]:
#LOAD BEST PERFORMING MODEL
model_path = "/content/drive/MyDrive/Gaussian_Integral/le_classifier.pt"
model = ProteinClassifierTCNWithAttentionNoDilation(
    input_dim=HYPERPARAMS['input_dim'],
    hidden_dim=HYPERPARAMS['hidden_dim'],
    output_dim=HYPERPARAMS['output_dim'],
    kernel_size=HYPERPARAMS['kernel_size']
)
model.load_state_dict(torch.load(model_path))
model.eval()


ProteinClassifierTCNWithAttentionNoDilation(
  (conv1): Conv1d(1, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu1): ReLU()
  (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu2): ReLU()
  (conv3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu3): ReLU()
  (conv4): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu4): ReLU()
  (attention): AttentionLayer(
    (attention_weights): Linear(in_features=128, out_features=1, bias=True)
    (softmax): Softmax(dim=1)
  )
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [ ]:
import json
with open("/content/drive/MyDrive/Gaussian_Integral/folds_gaussian_integral_ids_clean.json", "r") as f:
    folds_dict = json.load(f)

test_ids = folds_dict["test_ids"]
print(f"Total test IDs: {len(test_ids)}")


Total test IDs: 275


In [ ]:
# Asegurate de que la columna 1 contenga los IDs
data_ids = data.iloc[:, 1]
# test_ids ya lo tenés cargado desde el JSON
test_indices = data[data[1].isin(test_ids)].index
X_test = data.iloc[test_indices, 3:33].values  # Features: columnas 3 a 32
y_test = data.iloc[test_indices, 34].values    # Label: columna 34

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch
X_test_t = torch.from_numpy(X_test).unsqueeze(1)  # (N, 1, 30) para Conv1d
y_test_t = torch.from_numpy(y_test).unsqueeze(1)  # (N, 1)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t),
                         batch_size=HYPERPARAMS['batch_size'],
                         shuffle=False)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# 1. Definir device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")

# 2. Asegurar dtypes
model = model.float()
model.to(device).eval()

all_probs, all_preds, all_labels = [], [], []

# 3. Evaluación
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device, dtype=torch.float32)   # fuerza float32
        yb = yb.to(device, dtype=torch.float32)

        probs = model(xb).squeeze(1)              # (B,)
        preds = (probs >= 0.5).long()

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(yb.squeeze(1).cpu().numpy())

# 4. Concatenar resultados
all_probs = np.concatenate(all_probs)
all_preds = np.concatenate(all_preds).astype(int)
all_labels = np.concatenate(all_labels).astype(int)

# 5. Métricas
acc = accuracy_score(all_labels, all_preds)
rocauc = roc_auc_score(all_labels, all_probs)
report = classification_report(all_labels, all_preds, digits=3)
cm = confusion_matrix(all_labels, all_preds)

# 6. Mostrar resultados
print(f"✅ Test Accuracy: {acc:.3f}")
print(f"✅ Test ROC AUC: {rocauc:.3f}")
print("📄 Classification Report:\n", report)
print("📊 Confusion Matrix:\n", cm)




Usando device: cuda
✅ Test Accuracy: 0.720
✅ Test ROC AUC: 0.772
📄 Classification Report:
               precision    recall  f1-score   support

           0      0.703     0.750     0.726       136
           1      0.738     0.691     0.714       139

    accuracy                          0.720       275
   macro avg      0.721     0.720     0.720       275
weighted avg      0.721     0.720     0.720       275

📊 Confusion Matrix:
 [[102  34]
 [ 43  96]]
